# Gaussian mean and scale controls

When the data-only specification search (`mb.discover`) fits a Gaussian model, it decides whether the standard deviation (the scale) depends on the predictors, and it calibrates the prediction interval on held-out rows. This development study asks two questions. Does a standardized calibration score, $|Y-\mu|/\sigma$, give narrower 90% intervals at the same coverage than the default calibration? And can the comparison of scale candidates be arranged so that a varying scale cannot win only by changing the mean fit, which reports a varying scale when the mean is poorly fitted? A standardized-score experiment holds every fitted mean and scale fixed. Three conditional-scale prototypes then compare scale candidates with the mean fit shared across them. None met the development criteria fixed before measurement, so the Gaussian default and the rule for adopting a varying scale are unchanged, and no confirmation sample was drawn for these candidates.

Each process has ten independent development draws, 1,000 analysis observations, a common 500/500 fitting/calibration split and 2,000 independent evaluation observations. All contrasts reuse the same per-process inputs. Processes share random streams and are never pooled for uncertainty. The 1,900 method/draw records include 100 oracle-only diagnostics; they are not 1,900 independent datasets.

Five independent Uniform(−1,1) predictors are observed. Smooth means are $\sin(2\pi x_1)+0.5x_2$; sharp means are $3\operatorname{sign}(x_1-a)\operatorname{sign}(x_2-b)+x_3$ at $(a,b)=(0,0)$ or $(0.25,-0.2)$; the null mean is zero. Constant sharp standard deviation is 0.5, other constant controls use 1, and varying standard deviations are $0.3+1.5|x_1|$, $0.3+1.5\exp(-8x_1^2)$ or $\exp(0.7x_1)$. Errors are Gaussian except the constant-variance heavy-tail control, Student-t3 divided by $\sqrt{3}$. Extra columns are noise where the generating mean/scale omit them.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
from benches.evidence_float_codec import decode_record

base = root / "docs/book/benchmarks"
directory = base / "gaussian_scale_calibration"
rows = decode_record(json.loads((directory / "development_summary.json").read_text()))["rows"]
gate = json.loads((directory / "development_gate.json").read_text())
conditional = json.loads((directory / "CONDITIONAL_DEVELOPMENT_SUMMARY.json").read_text())

In [2]:
def headline(records, title):
    best = {}
    for panel in {r["panel"] for r in records}:
        eligible = [r for r in records if r["panel"] == panel and r.get("coverage_eligible") and not r["failures"]]
        for key in ("width", "true_mean_mse", "interval_score"):
            best[panel, key] = min((r[key] for r in eligible), default=None)

    def value(record, key):
        if key not in record:
            return "diagnostic only"
        text = f"{record[key]:.3f}"
        if key == "width" and record.get(key + "_se") is not None:
            text += f" ± {record[key + '_se']:.3f}"
        if record.get("coverage_eligible") and record[key] == best.get((record["panel"], key)):
            text = "**" + text + "**"
        return text

    frame = pl.DataFrame([{"Process": r["panel"], "Method": r["method"].replace("_", " "),
        "Width": value(r,"width"), "Coverage": value(r,"coverage"), "Mean MSE": value(r,"true_mean_mse"),
        "Interval score": value(r,"interval_score"), "Failures": r["failures"]} for r in records])
    # Coverage eligibility prevents a narrow under-covering procedure from being marked best.
    return (GT(frame,rowname_col="Method",groupname_col="Process").tab_stubhead("Method")
        .tab_header(title=title,subtitle="Ten independent draws per process; smaller width, mean MSE and score are better")
        .fmt_markdown(columns=["Width","Mean MSE","Interval score"])
        .tab_source_note(md("Width is real response-unit length, mean ± Monte Carlo SE across draws. Mean MSE compares "
            "the reported point with the generating mean, in squared response units. CQR points are pre-calibration "
            "band midpoints. Score is the 90% endpoint interval-score expression. Bold marks complete eligible "
            "minima within process, not a chosen new method. Oracle residual diagnostics produce no prediction band. "
            "Source: development_summary.json, which summarize_gaussian_scale_calibration.py computes from every "
            "complete per-draw archive; no successful-only aggregates.")))

In [3]:
headline([r for r in rows if r["panel"] == "smooth_bell"], "Every fixed method on bell-scale development")

GT(_tbl_data=shape: (16, 7)
┌─────────────┬────────────────┬────────────────┬──────────┬───────────┬────────────────┬──────────┐
│ Process     ┆ Method         ┆ Width          ┆ Coverage ┆ Mean MSE  ┆ Interval score ┆ Failures │
│ ---         ┆ ---            ┆ ---            ┆ ---      ┆ ---       ┆ ---            ┆ ---      │
│ str         ┆ str            ┆ str            ┆ str      ┆ str       ┆ str            ┆ i64      │
╞═════════════╪════════════════╪════════════════╪══════════╪═══════════╪════════════════╪══════════╡
│ smooth_bell ┆ discovery      ┆ 2.561 ± 0.031  ┆ 0.899    ┆ 0.021     ┆ 3.248          ┆ 0        │
│             ┆ central        ┆                ┆          ┆           ┆                ┆          │
│ smooth_bell ┆ discovery      ┆ 2.611 ± 0.041  ┆ 0.900    ┆ 0.021     ┆ 3.256          ┆ 0        │
│             ┆ normalized     ┆                ┆          ┆           ┆                ┆          │
│ smooth_bell ┆ supplied fixed ┆ 2.732 ± 0.041  ┆ 0.900    ┆ 0.130     ┆ 3.457          ┆ 0        │
│             ┆ central        ┆                ┆          ┆           ┆                ┆          │
│ smooth_bell ┆ supplied fixed ┆ 2.758 ± 0.048  ┆ 0.902    ┆ 0.130     ┆ 3.462          ┆ 0        │
│             ┆ normalized     ┆                ┆          ┆           ┆                ┆          │
│ smooth_bell ┆ supplied tuned ┆ **2.542 ±      ┆ 0.896    ┆ **0.016** ┆ **3.234**      ┆ 0        │
│             ┆ central        ┆ 0.030**        ┆          ┆           ┆                ┆          │
│ …           ┆ …              ┆ …              ┆ …        ┆ …         ┆ …              ┆ …        │
│ smooth_bell ┆ catboost split ┆ 3.230 ± 0.066  ┆ 0.894    ┆ 0.146     ┆ 4.972          ┆ 0        │
│ smooth_bell ┆ catboost cqr   ┆ 3.064 ± 0.039  ┆ 0.898    ┆ 0.208     ┆ 4.261          ┆ 0        │
│ smooth_bell ┆ lightgbm split ┆ 3.441 ± 0.073  ┆ 0.898    ┆ 0.239     ┆ 5.167          ┆ 0        │
│ smooth_bell ┆ lightgbm cqr   ┆ 2.671 ± 0.039  ┆ 0.897    ┆ 0.104     ┆ 3.778          ┆ 0        │
│ smooth_bell ┆ sklearn cqr    ┆ 2.887 ± 0.056  ┆ 0.896    ┆ 0.222     ┆ 3.991          ┆ 0        │
└─────────────┴────────────────┴────────────────┴──────────┴───────────┴────────────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x1092f5eb0>, _boxhead=Boxhead([ColInfo(var='Process', type=<ColInfoTypeEnum.row_group: 3>, column_label='Process', column_align='left', column_width=None, stub_level=None), ColInfo(var='Method', type=<ColInfoTypeEnum.stub: 2>, column_label='Method', column_align='left', column_width=None, stub_level=0), ColInfo(var='Width', type=<ColInfoTypeEnum.default: 1>, column_label='Width', column_align='left', column_width=None, stub_level=None), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='left', column_width=None, stub_level=None), ColInfo(var='Mean MSE', type=<ColInfoTypeEnum.default: 1>, column_label='Mean MSE', column_align='left', column_width=None, stub_level=None), ColInfo(var='Interval score', type=<ColInfoTypeEnum.default: 1>, column_label='Interval score', column_align='left', column_width=None, stub_level=None), ColInfo(var='Failures', type=<ColInfoTypeEnum.default: 1>, column_label='Failures', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10a6ce720>, _spanners=Spanners([]), _heading=Heading(title='Every fixed method on bell-scale development', subtitle='Ten independent draws per process; smaller width, mean MSE and score are better', preheader=None), _stubhead='Method', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x108dc83b0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10aa3e7e0>, _source_notes=[Md(text='Width is real response-unit length, mean ± Monte Carlo SE across draws. Mean MSE compares the reported point with the generating mean, in squared response units. CQR points are pre-calibration band midpoints. Score is the 

## Calibration scores

The default distributional interval starts with $\mu\pm z_{0.95}\sigma$ and adds one calibration correction, in response units, to both endpoints. The standardized score is $|Y-\mu|/\sigma$, with endpoints $\mu\pm q\sigma$, where $q$ is the $\lceil (n_c+1)\,0.9\rceil$-th smallest of the $n_c$ calibration scores; its correction is dimensionless and scales with $\sigma$. Positive finite scales are required. Both constructions calibrate a model fitted on the 500 fitting rows with the 500 calibration rows, and give marginal 90% coverage when rows are exchangeable; neither identifies the true noise mechanism.

Four fixed fitting tracks supply paired scores: ordinary discovery; a supplied varying-scale Gaussian location-scale (GaussianLSS) model at 100 updates per parameter; the same supplied formulas with the number of updates, equal for both parameters, chosen by five-fold cross-validation on the fitting rows over a grid that starts at 100 and is limited to 400; and a supplied constant-scale diagnostic. Smooth supplied mean is `bbs(x1,df=4)+bols(x2)`, sharp supplied mean is `btree(x1, x2, maxdepth=3, selection=sse) + bols(x3)`, and null mean is `1`. Varying supplied scale is `bbs(x1,df=4)`, constant scale `1`. Rate is 0.1. Every score pair reuses exactly the same fitted mean, scale and point vector.

The other fixed methods are discovery with the varying scale disabled, conformalized quantile regression (CQR) with the supplied mean formula at 1,600 updates and rate 0.1, a diagnostic that uses the true mean on the fitting rows (labelled oracle, excluded from comparisons), and five gradient-boosting references: split-conformal intervals (absolute-residual score) and CQR from CatBoost (500 trees, depth 6, rate 0.05) and from LightGBM (300 trees, 15 leaves, rate 0.05), and CQR from scikit-learn gradient boosting (300 trees, depth 3, rate 0.05).

The development criterion for the standardized score: on the bell and monotone smooth processes, its paired percent width excess over LightGBM CQR must have a one-sided 95% upper bound of at most 1%, with marginal coverage inside the eligibility band [0.88, 0.92], that is, within 0.02 of the nominal 0.90.

In [4]:
required = [r for r in rows if r["panel"] in ("smooth_bell","smooth_monotone")
            and r["method"] in ("discovery_normalized","supplied_fixed_normalized")]
frame = pl.DataFrame([{"Process":r["panel"],"Method":r["method"],"Coverage":r["coverage"],
    "Width excess %":r["width_excess"],"Upper 95%":r["width_upper95"],"Upper target":r["upper_target"]}
    for r in required])
(GT(frame,rowname_col="Method",groupname_col="Process").tab_stubhead("Required information track")
 .tab_header(title="Standardized-score development gate",subtitle="Paired percent width excess over prespecified LightGBM CQR; upper target ≤ 1%")
 .fmt_number(columns=["Coverage","Width excess %","Upper 95%"],decimals=3)
 .tab_source_note("Coverage eligibility is [0.88, 0.92] at nominal 0.90. The upper bound uses Student-t across ten paired "
                  "independent draws; these remain development results. All five reference comparisons and adjusted bounds are retained."))

GT(_tbl_data=shape: (4, 6)
┌─────────────────┬─────────────────────────┬──────────┬────────────────┬───────────┬──────────────┐
│ Process         ┆ Method                  ┆ Coverage ┆ Width excess % ┆ Upper 95% ┆ Upper target │
│ ---             ┆ ---                     ┆ ---      ┆ ---            ┆ ---       ┆ ---          │
│ str             ┆ str                     ┆ f64      ┆ f64            ┆ f64       ┆ bool         │
╞═════════════════╪═════════════════════════╪══════════╪════════════════╪═══════════╪══════════════╡
│ smooth_bell     ┆ discovery_normalized    ┆ 0.8998   ┆ -2.065303      ┆ 1.819789  ┆ false        │
│ smooth_bell     ┆ supplied_fixed_normaliz ┆ 0.9018   ┆ 3.410149       ┆ 7.2539    ┆ false        │
│                 ┆ ed                      ┆          ┆                ┆           ┆              │
│ smooth_monotone ┆ discovery_normalized    ┆ 0.90055  ┆ -2.701752      ┆ -0.372525 ┆ true         │
│ smooth_monotone ┆ supplied_fixed_normaliz ┆ 0.8982   ┆ 0.477954       ┆ 2.842436  ┆ false        │
│                 ┆ ed                      ┆          ┆                ┆           ┆              │
└─────────────────┴─────────────────────────┴──────────┴────────────────┴───────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x10aa3ee70>, _boxhead=Boxhead([ColInfo(var='Process', type=<ColInfoTypeEnum.row_group: 3>, column_label='Process', column_align='left', column_width=None, stub_level=None), ColInfo(var='Method', type=<ColInfoTypeEnum.stub: 2>, column_label='Method', column_align='left', column_width=None, stub_level=0), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='right', column_width=None, stub_level=None), ColInfo(var='Width excess %', type=<ColInfoTypeEnum.default: 1>, column_label='Width excess %', column_align='right', column_width=None, stub_level=None), ColInfo(var='Upper 95%', type=<ColInfoTypeEnum.default: 1>, column_label='Upper 95%', column_align='right', column_width=None, stub_level=None), ColInfo(var='Upper target', type=<ColInfoTypeEnum.default: 1>, column_label='Upper target', column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x108fcff80>, _spanners=Spanners([]), _heading=Heading(title='Standardized-score development gate', subtitle='Paired percent width excess over prespecified LightGBM CQR; upper target ≤ 1%', preheader=None), _stubhead='Required information track', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10aa81dc0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10aa81d00>, _source_notes=['Coverage eligibility is [0.88, 0.92] at nominal 0.90. The upper bound uses Student-t across ten paired independent draws; these remain development results. All five reference comparisons and adjusted bounds are retained.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10aa81b50>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10aa3f2c0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Aria

In [5]:
assert not gate["targets_pass"] and not gate["promote_default"] and not gate["confirmation_generated"]
print("The standardized-score candidate does not meet the development criterion and is not the default.")

The standardized-score candidate does not meet the development criterion and is not the default.


The standardized correction increases width on the bell control at comparable marginal coverage. The gate fails in the supplied fixed track and the bell data-only upper bound. The standardized score is available as `score="gaussian_normalized"` in `conformal_interval` (Python and R, GaussianLSS only, computed in the Rust core); automatic intervals for Gaussian models use the default calibration, which adds one correction in response units to both endpoints of $\mu\pm z_{0.95}\sigma$.

## Public score comparison

The example refits one development draw of the bell process with the supplied formulas and computes both intervals from the same fit. It adds no evidence to the comparison. Fitting, nuisance estimation and stopping choices precede calibration.

In [6]:
import mboostrs as mb
assert Path(mb.__file__).resolve().is_relative_to(root)
from benches.study_sharp_mean_controls import draw
from benches.study_discovery_calibration import partition

analysis, _, _ = draw("smooth_bell", 6293100, 1000)
evaluation, _, _ = draw("smooth_bell", 6393100, 2000)
fitting, calibration, partition_record = partition(analysis, 6293100)
model = mb.gamboostLSS({"mu":"y ~ bbs(x1, df=4) + bols(x2)","sigma":"y ~ bbs(x1, df=4)"},
    fitting,family=mb.GaussianLSS(),control=mb.boost_control(100))
legacy = mb.conformal_interval(model,evaluation.drop("y"),calibration=calibration,
                               score="absolute",extrapolation="clip")
standardized = mb.conformal_interval(model,evaluation.drop("y"),calibration=calibration,
                                     score="gaussian_normalized",extrapolation="clip")
np.testing.assert_array_equal(legacy.fitted_mean,standardized.fitted_mean)
comparison = pl.DataFrame({"Score":["Central band + additive","Standardized"],
    "Width":[float(legacy.width.mean()),float(standardized.width.mean())],
    "Coverage":[float(legacy.contains(evaluation["y"]).mean()),float(standardized.contains(evaluation["y"]).mean())]})
(GT(comparison).tab_header(title="Same fitted model, two calibration scores",subtitle="One repeated development input; response units")
 .fmt_number(columns=["Width","Coverage"],decimals=3)
 .tab_source_note("Mean predictions are identical. This display illustrates the transformation; it does not select a score or promote a default."))

GT(_tbl_data=shape: (2, 3)
┌─────────────────────────┬──────────┬──────────┐
│ Score                   ┆ Width    ┆ Coverage │
│ ---                     ┆ ---      ┆ ---      │
│ str                     ┆ f64      ┆ f64      │
╞═════════════════════════╪══════════╪══════════╡
│ Central band + additive ┆ 2.722946 ┆ 0.892    │
│ Standardized            ┆ 2.661374 ┆ 0.884    │
└─────────────────────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x1122462a0>, _boxhead=Boxhead([ColInfo(var='Score', type=<ColInfoTypeEnum.default: 1>, column_label='Score', column_align='left', column_width=None, stub_level=None), ColInfo(var='Width', type=<ColInfoTypeEnum.default: 1>, column_label='Width', column_align='right', column_width=None, stub_level=None), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x111d8e0f0>, _spanners=Spanners([]), _heading=Heading(title='Same fitted model, two calibration scores', subtitle='One repeated development input; response units', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1121905c0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x111007440>, _source_notes=['Mean predictions are identical. This display illustrates the transformation; it does not select a score or promote a default.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10aa3f290>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1092f6000>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2

## Conditional scale comparisons

When a constant-scale and a varying-scale location-scale model are each fitted jointly and compared, the varying-scale model also refits the mean. Under an incompletely fitted sharp mean, a scale term can therefore improve the out-of-fold score without describing a real change in the noise variance. With the current discovery, a varying scale is selected on three of ten draws of the axis-boundary process with constant true scale and on two of ten draws of the shifted-boundary process with constant true scale, and on none of the draws of the corresponding sharp processes whose scale does vary.

Version 1 fits one Gaussian mean at 200 updates on each fitting fold and shares its predictions across constant and nominated scale candidates. It fits absolute mean residuals with location fixed at zero and 200 scale updates, using Gaussian likelihood for the scale comparison. Symmetry makes signed and absolute residuals equivalent at zero location. Adoption uses the existing rule (the best candidate, adopted only when it is more than two standard errors better than the constant scale), and the final model is fitted jointly as before.

Version 2 chooses the number of updates of each outer-fold Gaussian mean by five-fold cross-validation inside the fold (grid from 100, limited to 400), and nominates scale candidates from the out-of-fold residuals. It shares those mean fits across scale candidates. On the ten null draws the prototype refused to tune an intercept-only mean; the predictions it returned instead are included, and the scale decision is unavailable there.

Version 3 guards intercept-only means and uses a fixed Student-t reference with four degrees of freedom for conditional scale fitting and scoring. Location remains zero and degrees of freedom remain four; only scale is fitted. This bounds the scale gradient's leverage from very large mean residuals. Returned prediction models still use the existing Gaussian pipeline after a supported adoption. The three prototypes are benchmark code, not part of the library.

In [7]:
versions = {"conditional_scale_comparison":"Fixed mean / Gaussian scale",
            "conditional_scale_comparison_v2":"Cross-fitted tuned mean",
            "conditional_scale_comparison_v3":"Cross-fitted mean / t4 scale"}
records = []
for r in rows:
    if r["method"] == "discovery_central":
        records.append({"Process":r["panel"],"Method":"Current discovery","Scale selected":f"{r['scale_adoptions']}/10",
            "Stage refusals":0,"Mean MSE":r["true_mean_mse"],"Width":r["width"],"Coverage":r["coverage"]})
for r in conditional:
    evaluated = not r["prediction_failures"] and not r["spread_refusals"]
    records.append({"Process":r["panel"],"Method":versions[r["version"]],
        "Scale selected":f"{r['adoptions']}/10" if evaluated else "diagnostic unavailable",
        "Stage refusals":r["spread_refusals"],"Mean MSE":r.get("true_mean_mse"),"Width":r.get("width"),"Coverage":r.get("coverage")})
(GT(pl.DataFrame(records),rowname_col="Method",groupname_col="Process").tab_stubhead("Comparison")
 .tab_header(title="Scale decisions and delivered predictions",subtitle="Ten paired development draws per process; smaller MSE/width are better at eligible coverage")
 .fmt_number(columns=["Mean MSE","Width","Coverage"],decimals=3)
 .tab_source_note("Scale selected means a varying-scale term was adopted. On constant-variance processes it is a false "
                  "structural attribution, not necessarily a prediction regression. Prediction metrics include every returned "
                  "fallback; they are not successful-only averages. Version 2 null scale decisions are unavailable because "
                  "all ten diagnostic stages refused. Counts are descriptive development outcomes, not population error bounds."))

GT(_tbl_data=shape: (40, 7)
┌──────────────────┬─────────────────┬──────────┬─────────────────┬──────────┬──────────┬──────────┐
│ Process          ┆ Method          ┆ Scale    ┆ Stage refusals  ┆ Mean MSE ┆ Width    ┆ Coverage │
│ ---              ┆ ---             ┆ selected ┆ ---             ┆ ---      ┆ ---      ┆ ---      │
│ str              ┆ str             ┆ ---      ┆ i64             ┆ f64      ┆ f64      ┆ f64      │
│                  ┆                 ┆ str      ┆                 ┆          ┆          ┆          │
╞══════════════════╪═════════════════╪══════════╪═════════════════╪══════════╪══════════╪══════════╡
│ axis_constant    ┆ Current         ┆ 3/10     ┆ 0               ┆ 1.570123 ┆ 3.165399 ┆ 0.89505  │
│                  ┆ discovery       ┆          ┆                 ┆          ┆          ┆          │
│ axis_varying     ┆ Current         ┆ 0/10     ┆ 0               ┆ 0.552127 ┆ 4.394486 ┆ 0.8984   │
│                  ┆ discovery       ┆          ┆                 ┆          ┆          ┆          │
│ shifted_constant ┆ Current         ┆ 2/10     ┆ 0               ┆ 0.50281  ┆ 2.289997 ┆ 0.89705  │
│                  ┆ discovery       ┆          ┆                 ┆          ┆          ┆          │
│ shifted_varying  ┆ Current         ┆ 0/10     ┆ 0               ┆ 0.432056 ┆ 4.306312 ┆ 0.9032   │
│                  ┆ discovery       ┆          ┆                 ┆          ┆          ┆          │
│ smooth_constant  ┆ Current         ┆ 0/10     ┆ 0               ┆ 0.035723 ┆ 3.342674 ┆ 0.89625  │
│                  ┆ discovery       ┆          ┆                 ┆          ┆          ┆          │
│ …                ┆ …               ┆ …        ┆ …               ┆ …        ┆ …        ┆ …        │
│ shifted_varying  ┆ Cross-fitted    ┆ 10/10    ┆ 0               ┆ 0.39212  ┆ 3.846282 ┆ 0.89595  │
│                  ┆ mean / t4 scale ┆          ┆                 ┆          ┆          ┆          │
│ smooth_absolute  ┆ Cross-fitted    ┆ 10/10    ┆ 0               ┆ 0.061527 ┆ 3.556025 ┆ 0.8964   │
│                  ┆ mean / t4 scale ┆          ┆                 ┆          ┆          ┆          │
│ smooth_bell      ┆ Cross-fitted    ┆ 10/10    ┆ 0               ┆ 0.020937 ┆ 2.561479 ┆ 0.8986   │
│                  ┆ mean / t4 scale ┆          ┆                 ┆          ┆          ┆          │
│ smooth_constant  ┆ Cross-fitted    ┆ 0/10     ┆ 0               ┆ 0.035723 ┆ 3.342674 ┆ 0.89625  │
│                  ┆ mean / t4 scale ┆          ┆                 ┆          ┆          ┆          │
│ smooth_monotone  ┆ Cross-fitted    ┆ 10/10    ┆ 0               ┆ 0.063171 ┆ 3.657913 ┆ 0.90025  │
│                  ┆ mean / t4 scale ┆          ┆                 ┆          ┆          ┆          │
└──────────────────┴─────────────────┴──────────┴─────────────────┴──────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x11222d550>, _boxhead=Boxhead([ColInfo(var='Process', type=<ColInfoTypeEnum.row_group: 3>, column_label='Process', column_align='left', column_width=None, stub_level=None), ColInfo(var='Method', type=<ColInfoTypeEnum.stub: 2>, column_label='Method', column_align='left', column_width=None, stub_level=0), ColInfo(var='Scale selected', type=<ColInfoTypeEnum.default: 1>, column_label='Scale selected', column_align='left', column_width=None, stub_level=None), ColInfo(var='Stage refusals', type=<ColInfoTypeEnum.default: 1>, column_label='Stage refusals', column_align='right', column_width=None, stub_level=None), ColInfo(var='Mean MSE', type=<ColInfoTypeEnum.default: 1>, column_label='Mean MSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='Width', type=<ColInfoTypeEnum.default: 1>, column_label='Width', column_align='right', column_width=None, stub_level=None), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x111005730>, _

The robust conditional comparison selects no varying scale on the tested constant-variance controls and detects all smooth varying-scale controls. Its axis-boundary control with varying scale adopts a varying scale on only five of ten draws; the criterion fixed in advance required adoption on at least 90% of draws of every varying-scale process. The candidate is therefore not adopted. Sharing the mean fit removes one source of false scale reports but does not make the scale decision reliable under a sharp mean.

## Sharp mean with a smooth scale: a stated limitation

On two processes from one generator, the default did not detect a smooth scale under a sharp mean. Both processes use the scale 0.3 + 1.5|x1|, the same scale as the smooth_absolute control. Their means are 3·sign(x1)·sign(x2) + x3 (axis_varying) and 3·sign(x1−0.25)·sign(x2+0.2) + x3 (shifted_varying). In the sharp-mean confirmation battery on reserved seeds 6294100–6294139 (40 draws per process), discovery adopted a varying scale on 0 of 40 draws of axis_varying and 0 of 40 draws of shifted_varying, while adopting on 40 of 40 draws of each smooth varying-scale process (absolute, bell and monotone) and on 0 of 40 draws of every constant-variance process. In the ten development draws, run without the mean-adequacy guard (a check that the mean fit is adequate before a scale is considered), the same sharp processes also adopted on 0 of 10, so the guard does not cause the miss. A diagnostic recorded with the decision shows the failure at the nomination and screening steps, on residuals of the 200-update screening mean fit; the scale term that fails there is the same smooth term the smooth processes adopt.

This is an accepted limitation of the default, decided in the [sharp-scale decision record](../appendix/records/spread_and_distributional.md). Marginal coverage on the two sharp varying-scale processes is 0.901, inside the [0.88, 0.92] eligibility band; what is lost is scale attribution, and coverage split by true-scale tercile is unbalanced (0.963 low / 0.805 high on axis_varying; 0.978 / 0.793 on shifted_varying, against a nominal 0.90). A supplied GaussianLSS sigma formula, or a spread column that discovery is required to use, fits a varying scale on such data. A claim that the default recovers a smooth scale under a sharp mean would need a candidate fixed on development draws and confirmed on fresh draws, under criteria stated in advance in the decision record.

## Complete score controls

In [8]:
headline(rows, "All fixed score-development methods and processes")

GT(_tbl_data=shape: (160, 7)
┌───────────────┬────────────────┬───────────────┬──────────┬──────────┬────────────────┬──────────┐
│ Process       ┆ Method         ┆ Width         ┆ Coverage ┆ Mean MSE ┆ Interval score ┆ Failures │
│ ---           ┆ ---            ┆ ---           ┆ ---      ┆ ---      ┆ ---            ┆ ---      │
│ str           ┆ str            ┆ str           ┆ str      ┆ str      ┆ str            ┆ i64      │
╞═══════════════╪════════════════╪═══════════════╪══════════╪══════════╪════════════════╪══════════╡
│ axis_constant ┆ discovery      ┆ 3.165 ± 0.595 ┆ 0.895    ┆ 1.570    ┆ 4.306          ┆ 0        │
│               ┆ central        ┆               ┆          ┆          ┆                ┆          │
│ axis_constant ┆ discovery      ┆ 3.166 ± 0.596 ┆ 0.897    ┆ 1.570    ┆ 4.280          ┆ 0        │
│               ┆ normalized     ┆               ┆          ┆          ┆                ┆          │
│ axis_constant ┆ supplied fixed ┆ 3.681 ± 0.632 ┆ 0.900    ┆ 2.066    ┆ 4.804          ┆ 0        │
│               ┆ central        ┆               ┆          ┆          ┆                ┆          │
│ axis_constant ┆ supplied fixed ┆ 3.697 ± 0.627 ┆ 0.902    ┆ 2.066    ┆ 4.789          ┆ 0        │
│               ┆ normalized     ┆               ┆          ┆          ┆                ┆          │
│ axis_constant ┆ supplied tuned ┆ 3.631 ± 0.477 ┆ 0.904    ┆ 1.532    ┆ 4.758          ┆ 0        │
│               ┆ central        ┆               ┆          ┆          ┆                ┆          │
│ …             ┆ …              ┆ …             ┆ …        ┆ …        ┆ …              ┆ …        │
│ heavy_tail    ┆ catboost split ┆ 3.057 ± 0.057 ┆ 0.902    ┆ 0.179    ┆ 4.724          ┆ 0        │
│ heavy_tail    ┆ catboost cqr   ┆ 3.832 ± 0.079 ┆ 0.912    ┆ 0.382    ┆ 5.270          ┆ 0        │
│ heavy_tail    ┆ lightgbm split ┆ 3.261 ± 0.067 ┆ 0.902    ┆ 0.264    ┆ 4.831          ┆ 0        │
│ heavy_tail    ┆ lightgbm cqr   ┆ 3.291 ± 0.057 ┆ 0.907    ┆ 0.215    ┆ 4.783          ┆ 0        │
│ heavy_tail    ┆ sklearn cqr    ┆ 3.478 ± 0.039 ┆ 0.911    ┆ 0.358    ┆ 4.951          ┆ 0        │
└───────────────┴────────────────┴───────────────┴──────────┴──────────┴────────────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x112732300>, _boxhead=Boxhead([ColInfo(var='Process', type=<ColInfoTypeEnum.row_group: 3>, column_label='Process', column_align='left', column_width=None, stub_level=None), ColInfo(var='Method', type=<ColInfoTypeEnum.stub: 2>, column_label='Method', column_align='left', column_width=None, stub_level=0), ColInfo(var='Width', type=<ColInfoTypeEnum.default: 1>, column_label='Width', column_align='left', column_width=None, stub_level=None), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='left', column_width=None, stub_level=None), ColInfo(var='Mean MSE', type=<ColInfoTypeEnum.default: 1>, column_label='Mean MSE', column_align='left', column_width=None, stub_level=None), ColInfo(var='Interval score', type=<ColInfoTypeEnum.default: 1>, column_label='Interval score', column_align='left', column_width=None, stub_level=None), ColInfo(var='Failures', type=<ColInfoTypeEnum.default: 1>, column_label='Failures', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x112751f10>, _spanners=Spanners([]), _heading=Heading(title='All fixed score-development methods and processes', subtitle='Ten independent draws per process; smaller width, mean MSE and score are better', preheader=None), _stubhead='Method', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11279d6d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11279d5e0>, _source_notes=[Md(text='Width is real response-unit length, mean ± Monte Carlo SE across draws. Mean MSE compares the reported point with the generating mean, in squared response units. CQR points are pre-calibration band midpoints. Score i

## Evidence and reproduction

Each protocol and its runner were fixed before measurement; the source identities are recorded in the validation files. The source validator verifies all 1,900 archive and runner identities and identical per-process input hashes. The model-fitting Rust code is the same in all contrasts; the standardized-score functions are recorded separately in the native build identity.

All score-study calibration calls and competitor predictions/scores are retained. Conditional version 1 keeps the fold means, scale predictions, scores and returned intervals; versions 2 and 3 also keep every calibration call. Some fits in the conditional runs logged refusals of an optional tree refit; those refusals are kept in the logs and did not affect a scale decision.

The default limit on boosting updates is 400 for a mean model and 1,600 per parameter for an LSS model ([stopping limits](default_stopping_ceilings.md)); the supplied tuned controls use an explicit limit of 400. The versions of the external libraries are listed in `ENVIRONMENT_ADDENDUM.json`.

Checks of the standardized score cover inversion, the boundary of positive scales, invariance under affine changes of the response, agreement between internal and external calibration, and the routing of the default score. On fixed inputs, Python and R give identical scores and bands; fitted values differ by at most $8.88\times10^{-16}$. These checks establish that the implementations agree, not that the score is efficient.

From the repository root, with the recorded source version:

```bash
PYTHONPATH=python:. .venv/bin/python benches/study_gaussian_scale_calibration.py \
  --phase development --panels smooth_bell --output /tmp/gaussian-score-bell.jsonl.xz
PYTHONPATH=python:. .venv/bin/python benches/summarize_gaussian_scale_calibration.py --phase development
PYTHONPATH=python:. .venv/bin/python benches/summarize_conditional_scale_development.py
```

The conditional runners accept `--panel` and a new `--output` filename. They refuse to overwrite existing results. Timings were measured with other jobs running and are descriptive; memory was not measured.

In [9]:
folders = ["gaussian_scale_calibration", "conditional_scale_comparison", "conditional_scale_comparison_v2", "conditional_scale_comparison_v3"]
# Per-draw archives are release assets listed in raw_archives.json; the bundle holds the files this page reads or cites.
artifacts = [folder+"/"+p.name for folder in folders for p in sorted((base/folder).iterdir())
             if p.is_file() and not p.name.endswith((".jsonl.xz", ".jsonl.gz"))]
(GT(pl.DataFrame({"Artifact":artifacts})).tab_header(title="Bundled development evidence",
    subtitle="Protocols, run metadata, summaries and validation")
 .tab_source_note("The four experiments reuse development inputs. The data download is not independent confirmation evidence. "
                  "Per-draw archives (*.jsonl.xz) are assets of the benchmark-raw-archives-1 GitHub release, listed in raw_archives.json, and are not bundled."))

Bundled development evidence
"Protocols, run metadata, summaries and validation"
Artifact
gaussian_scale_calibration/BASELINE_STAGE_VALIDATION.json
gaussian_scale_calibration/CONDITIONAL_DEVELOPMENT_SUMMARY.json
gaussian_scale_calibration/DEVELOPMENT_DECISION.md
gaussian_scale_calibration/ENVIRONMENT_ADDENDUM.json
gaussian_scale_calibration/PROTOCOL.md
gaussian_scale_calibration/R_PARITY_VALIDATION.json
gaussian_scale_calibration/SOURCE_VALIDATION.json
gaussian_scale_calibration/development_axis_constant.jsonl.metadata.json
